# Data Parallel Loading: Simulating DistributedSampler

**Difficulty**: 🟡 Medium

**Companies**: Meta, Google, NVIDIA, Anthropic

---

### Problem Statement

In distributed data-parallel training, each GPU (rank) must process a **disjoint shard** of the dataset. PyTorch's `DistributedSampler` handles this: it assigns indices to ranks so that (1) every sample is seen exactly once across all ranks per epoch, and (2) each rank gets the same number of samples (padding if needed).

Your task is to implement a simplified `DistributedSampler` from scratch — without `torch.distributed` — to understand the sharding and shuffling logic.

---

### Requirements

1. **`SimpleDistributedSampler(dataset, num_replicas, rank, shuffle, seed)`**:
   - `num_replicas`: total number of ranks (GPUs).
   - `rank`: this rank's index (0-based).
   - If the dataset size is not divisible by `num_replicas`, **pad** by repeating samples from the start so every rank gets the same count.
   - If `shuffle=True`, shuffle indices deterministically using the `seed + epoch` (call `set_epoch(epoch)` to change the epoch).
   - Implement `__iter__` and `__len__`.

2. **Verify sharding** — The union of indices from all ranks must equal the full dataset (with padding if needed). No rank should share indices with another.

3. **Verify shuffle reproducibility** — Same `seed + epoch` must give the same order. Different epochs must give different orders.

---

### Constraints

- ✅ Must work as a `sampler` argument to `DataLoader`.
- ✅ Every rank must get exactly `ceil(len(dataset) / num_replicas)` indices.
- ✅ Shuffling must be deterministic given seed + epoch.
- ❌ Do **not** use `torch.utils.data.distributed.DistributedSampler`.

---

<details>
  <summary>💡 Hint</summary>

  - Total padded size: `ceil(len(dataset) / num_replicas) * num_replicas`.
  - Pad by extending the index list with indices from the start: `indices += indices[:padding]`.
  - To shard: `indices[rank::num_replicas]` — every rank takes every `num_replicas`-th index.
  - For deterministic shuffle: create a `torch.Generator` with `manual_seed(seed + epoch)` and use `torch.randperm`.

</details>

---

In [ ]:
import math
import torch
from torch.utils.data import Dataset, DataLoader, Sampler

# Reuse a simple map-style dataset
class SimpleDataset(Dataset):
    def __init__(self, N):
        self.data = torch.arange(N, dtype=torch.float32)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

ds = SimpleDataset(103)  # intentionally not divisible by common GPU counts
print(f"Dataset size: {len(ds)}")

In [ ]:
class SimpleDistributedSampler(Sampler):
    """Simplified DistributedSampler: shards a dataset across ranks."""

    def __init__(self, dataset, num_replicas: int, rank: int,
                 shuffle: bool = True, seed: int = 0):
        self.dataset = dataset
        self.num_replicas = num_replicas
        self.rank = rank
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        # TODO: Compute num_samples (per rank) and total_size (padded)
        ...

    def set_epoch(self, epoch: int):
        """Set the epoch for deterministic shuffling."""
        self.epoch = epoch

    def __len__(self):
        # TODO: Return the number of samples this rank will yield
        ...

    def __iter__(self):
        # TODO: Build the full index list [0, 1, ..., len(dataset)-1]
        # TODO: If shuffle, use a seeded generator to permute indices
        # TODO: Pad indices to total_size by repeating from the start
        # TODO: Shard: take every num_replicas-th index starting at self.rank
        # TODO: Yield each index
        ...

In [ ]:
# === Tests ===
num_replicas = 4

# Test 1: All ranks get the same number of samples
samplers = [SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=r, shuffle=False)
            for r in range(num_replicas)]
lengths = [len(s) for s in samplers]
assert len(set(lengths)) == 1, f"Ranks got different lengths: {lengths}"
expected_per_rank = math.ceil(len(ds) / num_replicas)
assert lengths[0] == expected_per_rank, \
    f"Expected {expected_per_rank} per rank, got {lengths[0]}"
print(f"PASSED: All {num_replicas} ranks get {lengths[0]} samples each")

# Test 2: No overlap between ranks (disjoint shards)
all_indices = []
for s in samplers:
    indices = list(s)
    all_indices.append(indices)
flat = [i for shard in all_indices for i in shard]
assert len(flat) == expected_per_rank * num_replicas, "Total indices count is wrong"
# Every real index should appear at least once
real_indices = set(range(len(ds)))
assert real_indices.issubset(set(flat)), "Some dataset indices are missing"
print("PASSED: Shards are disjoint and cover the full dataset")

# Test 3: Works as DataLoader sampler
sampler_r0 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=False)
loader = DataLoader(ds, batch_size=8, sampler=sampler_r0)
loaded = torch.cat([b for b in loader])
assert len(loaded) == expected_per_rank, f"DataLoader yielded {len(loaded)}, expected {expected_per_rank}"
print("PASSED: Works with DataLoader")

# Test 4: Deterministic shuffling — same seed+epoch → same order
s1 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
s2 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
assert list(s1) == list(s2), "Same seed should give same order"
print("PASSED: Deterministic shuffle with same seed")

# Test 5: Different epochs → different order
s3 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
s3.set_epoch(1)
s4 = SimpleDistributedSampler(ds, num_replicas=num_replicas, rank=0, shuffle=True, seed=42)
s4.set_epoch(2)
assert list(s3) != list(s4), "Different epochs should give different orders"
print("PASSED: Different epochs produce different shuffles")

print("\nAll tests passed!")